In [2]:
# ============================================================
# FinPulse — Fixed Scoring Model (month-by-month KPIs)
# Replace your previous Cell 10 with this entire file
# Run AFTER finpulse_cleaning.py has produced the 3 CSVs
# ============================================================
 
import pandas as pd
import numpy as np
 
# ============================================================
# CELL 1 — Load cleaned data
# ============================================================
 
transactions = pd.read_csv('finpulse_transactions_clean.csv')
monthly      = pd.read_csv('finpulse_monthly_summary.csv')
 
# Make sure trans_date is parsed as datetime
transactions['trans_date'] = pd.to_datetime(transactions['trans_date'], errors='coerce')
transactions['year']  = transactions['trans_date'].dt.year
transactions['month'] = transactions['trans_date'].dt.month
 
print(f"Transactions: {transactions.shape}")
print(f"Monthly rows: {monthly.shape}")

Transactions: (1266862, 29)
Monthly rows: (18, 6)


In [3]:
# ============================================================
# CELL 2 — KPI weights
# ============================================================
 
WEIGHTS = {
    'savings_rate'       : 0.30,
    'spending_stability' : 0.20,
    'essential_ratio'    : 0.20,
    'avg_transaction'    : 0.15,
    'diversity_penalty'  : 0.15,
}
 
ESSENTIAL     = ['groceries', 'health', 'transport', 'home', 'kids & pets']
DISCRETIONARY = ['dining', 'entertainment', 'shopping', 'travel',
                 'personal care', 'miscellaneous']

In [4]:
# ============================================================
# CELL 3 — Score each month independently
# All 5 KPIs are recalculated using only that month's data
# ============================================================
 
# Estimate income = 1.3x the highest monthly spend (global, stays fixed)
estimated_income = monthly['total_spent'].max() * 1.3
 
records = []
 
for _, row in monthly.iterrows():
    yr  = int(row['year'])
    mo  = int(row['month'])
 
    # Slice transactions for this month only
    mask   = (transactions['year'] == yr) & (transactions['month'] == mo)
    month_txns = transactions[mask]
 
    if len(month_txns) == 0:
        continue
 
    total_spent = row['total_spent']
 
    # --- KPI 1: Savings Rate ---
    savings_rate  = max(0, (estimated_income - total_spent) / estimated_income)
    sr_score      = min(100, savings_rate / 0.20 * 100)
 
    # --- KPI 2: Spending Stability (rolling 3-month CV) ---
    # Use the 3 months around this month for a rolling window
    nearby = monthly[
        (monthly['year'] == yr) &
        (monthly['month'].between(mo - 2, mo))
    ]['total_spent']
    if len(nearby) > 1:
        cv = nearby.std() / nearby.mean()
    else:
        cv = 0  # only one month available, assume stable
    stab_score = max(0, min(100, (1 - cv / 0.5) * 100))
 
    # --- KPI 3: Essential vs Discretionary Ratio ---
    month_cats = month_txns.groupby('category_clean')['amt'].sum().reset_index()
    month_total = month_cats['amt'].sum()
 
    essential_spend = month_cats[
        month_cats['category_clean'].isin(ESSENTIAL)
    ]['amt'].sum()
 
    essential_ratio = essential_spend / month_total if month_total > 0 else 0
    essential_score = float(np.clip((essential_ratio - 0.30) / (0.60 - 0.30) * 100, 0, 100))
 
    # --- KPI 4: Average Transaction Size ---
    avg_txn     = month_txns['amt'].mean()
    avt_score   = float(np.clip((1 - (avg_txn - 10) / (150 - 10)) * 100, 0, 100))
 
    # --- KPI 5: Category Diversity ---
    if month_total > 0:
        max_cat_pct = month_cats['amt'].max() / month_total
    else:
        max_cat_pct = 0
    div_score = float(np.clip((1 - (max_cat_pct - 0.40) / (0.80 - 0.40)) * 100, 0, 100))
 
    # --- Final weighted score ---
    final = (
        sr_score       * WEIGHTS['savings_rate']       +
        stab_score     * WEIGHTS['spending_stability'] +
        essential_score* WEIGHTS['essential_ratio']    +
        avt_score      * WEIGHTS['avg_transaction']    +
        div_score      * WEIGHTS['diversity_penalty']
    )
 
    records.append({
        'year'              : yr,
        'month'             : mo,
        'month_label'       : row['month_label'],
        'total_spent'       : round(total_spent, 2),
        'estimated_income'  : round(estimated_income, 2),
        'savings_rate'      : round(savings_rate, 4),
        'kpi_savings'       : round(sr_score, 1),
        'kpi_stability'     : round(stab_score, 1),
        'kpi_essential'     : round(essential_score, 1),
        'kpi_avg_txn'       : round(avt_score, 1),
        'kpi_diversity'     : round(div_score, 1),
        'health_score'      : round(final, 1),
    })
 
scored = pd.DataFrame(records)

In [5]:
# ============================================================
# CELL 4 — Print the results (should vary month to month now)
# ============================================================
 
print("\n" + "="*75)
print(f"  {'Month':<12} {'Spent':>10} {'Savings':>8} {'Stab':>6} {'Ess':>6} {'AvgTxn':>7} {'Div':>6} {'SCORE':>7}")
print("="*75)
for _, r in scored.iterrows():
    print(
        f"  {r['month_label']:<12}"
        f"  ${r['total_spent']:>9,.0f}"
        f"  {r['kpi_savings']:>6.1f}"
        f"  {r['kpi_stability']:>6.1f}"
        f"  {r['kpi_essential']:>6.1f}"
        f"  {r['kpi_avg_txn']:>6.1f}"
        f"  {r['kpi_diversity']:>6.1f}"
        f"  {r['health_score']:>6.1f}"
    )
print("="*75)
print(f"\n  Avg score : {scored['health_score'].mean():.1f}")
print(f"  Min score : {scored['health_score'].min():.1f}  ({scored.loc[scored['health_score'].idxmin(), 'month_label']})")
print(f"  Max score : {scored['health_score'].max():.1f}  ({scored.loc[scored['health_score'].idxmax(), 'month_label']})")


  Month             Spent  Savings   Stab    Ess  AvgTxn    Div   SCORE
  Jan 2019      $2,846,897   100.0   100.0   100.0    67.4   100.0    95.1
  Feb 2019      $2,727,054   100.0    93.9   100.0    67.0   100.0    93.8
  Mar 2019      $3,851,718   100.0    60.7   100.0    67.5   100.0    87.3
  Apr 2019      $3,716,544   100.0    64.2   100.0    67.3   100.0    87.9
  May 2019      $3,943,308   100.0    94.1   100.0    67.4   100.0    93.9
  Jun 2019      $4,687,300   100.0    75.3   100.0    67.3   100.0    90.2
  Jul 2019      $4,693,026   100.0    80.6   100.0    67.5   100.0    91.2
  Aug 2019      $4,730,727   100.0    99.0   100.0    67.6   100.0    94.9
  Sep 2019      $3,811,363   100.0    76.4   100.0    67.7   100.0    90.4
  Oct 2019      $3,760,888   100.0    73.4   100.0    67.1   100.0    89.7
  Nov 2019      $3,836,184   100.0    98.0   100.0    67.3   100.0    94.7
  Dec 2019      $7,656,762   100.0    12.4   100.0    67.5   100.0    77.6
  Jan 2020      $2,866,078 

In [6]:
# ============================================================
# CELL 5 — Overall summary score (average across all months)
# ============================================================
 
overall_score = scored['health_score'].mean()
 
def get_grade(score):
    if score >= 80: return "A", "Excellent"
    if score >= 65: return "B", "Good"
    if score >= 50: return "C", "Fair"
    if score >= 35: return "D", "Poor"
    return "F", "Critical"
 
grade, label = get_grade(overall_score)
 
print(f"\n  Overall Financial Health Score : {overall_score:.1f}/100")
print(f"  Grade : {grade}  |  {label}")


  Overall Financial Health Score : 90.5/100
  Grade : A  |  Excellent


In [7]:
# ============================================================
# CELL 6 — Save outputs for Tableau
# ============================================================
 
scored.to_csv('finpulse_monthly_scores.csv', index=False)
print("\n✅ Saved: finpulse_monthly_scores.csv")
print("   Scores now vary month-to-month — ready for Tableau.")


✅ Saved: finpulse_monthly_scores.csv
   Scores now vary month-to-month — ready for Tableau.
